# Train CENTINAL's crowd counter on more data (Colab GPU)

Continues training the current CSRNet model (`csrnet_centinal.pth`) on four datasets:

| Dataset | Train images | Why |
|---|---|---|
| ShanghaiTech A + B | 300 + 400 | What the model was trained on so far; keeps sparse-crowd accuracy |
| **UCF-QNRF** | 1,201 | Very dense crowds, up to 12,865 people per image |
| **JHU-CROWD++** | 2,272 | Weather, night and lighting variation; up to ~25,000 per image |

The goal is to fix the model's biggest weakness: very large crowds (1,000+ people) are
still undercounted by about 114 people on average.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells in order.

- Checkpoints are saved to Google Drive, so a Colab disconnect loses nothing. If you
  get disconnected, rerun cells 1–4, then the **Resume** cell.
- The best checkpoint is chosen on *validation* images. Test images are only used at the end.
- JHU-CROWD++ is licensed for **non-commercial use only**.

## 1. Settings

In [ ]:
CODE_SOURCE = "github"          # or "upload" (see cell 3)
REPO_URL = "https://github.com/Daksh-Upadhayay/CENTINAL.git"
BRANCH = "improve-models-and-eval"

DATASETS = "sha,shb,qnrf,jhu"   # ShanghaiTech A, ShanghaiTech B, UCF-QNRF, JHU-CROWD++
INIT_FROM = "csrnet_centinal.pth"   # the current default model; training continues from it
LR = 3e-6
EPOCHS = 30
EVAL_EVERY = 1
TIME_BUDGET_MIN = 180           # training stops cleanly after this; evaluation runs afterwards

# Downloads total about 7.8 GB. Set True to keep a copy of the zips in Google Drive so a
# later session can skip re-downloading. Needs about 8 GB of free Drive space.
CACHE_ZIPS_IN_DRIVE = False

# Score the *old* model on UCF-QNRF and JHU-CROWD++ too, so the before/after comparison is
# fair. Adds roughly as long again to the evaluation step.
EVALUATE_OLD_MODEL = True

DRIVE_DIR = "/content/drive/MyDrive/centinal_training"
CHECKPOINT = f"{DRIVE_DIR}/csrnet_centinal_v2.pth"

## 2. Check the GPU and mount Drive

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoints will be saved to", DRIVE_DIR)

## 3. Get the code

**`github`** clones the branch, which must be pushed first:
`git push -u origin improve-models-and-eval`

**`upload`**: on your Mac, from the repo folder, run

```
zip -r centinal_colab_bundle.zip centinal train_csrnet.py train_lstm.py eval videos/crowd_test.mp4 csrnet_centinal.pth
```

then set `CODE_SOURCE = "upload"` in cell 1 and run this cell.

In [ ]:
import os
%cd /content
if CODE_SOURCE == "github":
    !rm -rf /content/CENTINAL
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/CENTINAL
else:
    from google.colab import files
    uploaded = files.upload()   # choose centinal_colab_bundle.zip
    !rm -rf /content/CENTINAL && mkdir -p /content/CENTINAL
    !unzip -q -o centinal_colab_bundle.zip -d /content/CENTINAL

if not os.path.exists("/content/CENTINAL/train_csrnet.py"):
    raise RuntimeError(
        "Code not found in /content/CENTINAL. If CODE_SOURCE is 'github', the branch "
        f"'{BRANCH}' is probably not pushed yet -- run `git push -u origin {BRANCH}` "
        "on your Mac, then rerun this cell. Otherwise use CODE_SOURCE = 'upload'.")
if "--datasets" not in open("/content/CENTINAL/train_csrnet.py").read():
    raise RuntimeError(
        "This copy of the code predates multi-dataset training. Push the latest commits "
        "on your Mac and rerun this cell.")

%cd /content/CENTINAL
!ls train_csrnet.py centinal/datasets.py {INIT_FROM}

## 4. Download the datasets (~7.8 GB, first run only per session)

Each dataset is downloaded, unzipped to `/content/data`, and checked. If a download
fails, the cell says what to do. The usual fix for JHU-CROWD++ is to put
`jhu_crowd_v2.0.zip` in `MyDrive/centinal_training/datasets/` yourself and rerun.

In [ ]:
import os, shutil, subprocess, sys, zipfile

DATA = "/content/data"
ZIPS = "/content/zips"
DRIVE_ZIPS = f"{DRIVE_DIR}/datasets"
os.makedirs(DATA, exist_ok=True); os.makedirs(ZIPS, exist_ok=True)

SOURCES = {
    "shanghaitech": ("ShanghaiTech_Crowd_Counting_Dataset.zip",
                     ["curl", "-L", "--fail", "-#", "-o", "{out}",
                      "https://www.dropbox.com/scl/fi/dkj5kulc9zj0rzesslck8/"
                      "ShanghaiTech_Crowd_Counting_Dataset.zip?rlkey=ymbcj50ac04uvqn8p49j9af5f&dl=1"]),
    # The UCF server's certificate chain is incomplete, hence -k.
    "qnrf": ("UCF-QNRF_ECCV18.zip",
             ["curl", "-kL", "--fail", "-#", "-o", "{out}",
              "https://www.crcv.ucf.edu/data/ucf-qnrf/UCF-QNRF_ECCV18.zip"]),
    "jhu": ("jhu_crowd_v2.0.zip",
            [sys.executable, "-m", "gdown", "1pA7ZeXU3hh-1txS9lFQiCek1ts3MdBaj", "-O", "{out}"]),
}
wanted = {"sha": "shanghaitech", "shb": "shanghaitech", "qnrf": "qnrf", "jhu": "jhu"}
needed = sorted({wanted[d.strip()] for d in DATASETS.split(",")})

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)

for source in needed:
    zip_name, command = SOURCES[source]
    marker = os.path.join(DATA, f".{source}_extracted")
    if os.path.exists(marker):
        print(f"{source}: already extracted"); continue

    local_zip = os.path.join(ZIPS, zip_name)
    cached_zip = os.path.join(DRIVE_ZIPS, zip_name)
    if os.path.exists(cached_zip):
        print(f"{source}: copying {zip_name} from Drive"); shutil.copy(cached_zip, local_zip)
    else:
        print(f"{source}: downloading {zip_name}")
        subprocess.run([part.format(out=local_zip) for part in command], check=False)

    if not (os.path.exists(local_zip) and zipfile.is_zipfile(local_zip)):
        raise RuntimeError(
            f"{source}: download failed or is not a zip. Download {zip_name} yourself, put it in "
            f"{DRIVE_ZIPS}/ (create the folder if needed), and rerun this cell."
            + ("\nJHU-CROWD++ mirrors: http://www.crowd-counting.com/#download" if source == "jhu" else ""))

    if CACHE_ZIPS_IN_DRIVE and not os.path.exists(cached_zip):
        os.makedirs(DRIVE_ZIPS, exist_ok=True); shutil.copy(local_zip, cached_zip)
        print(f"{source}: saved a copy to Drive")

    print(f"{source}: unzipping"); subprocess.run(["unzip", "-q", "-o", local_zip, "-d", DATA], check=True)
    os.remove(local_zip)
    open(marker, "w").close()

sys.path.insert(0, "/content/CENTINAL")
from centinal.datasets import DATASET_LABELS, dataset_splits
for name in [d.strip() for d in DATASETS.split(",")]:
    splits = dataset_splits(DATA, name)
    print(f"{DATASET_LABELS[name]:<15}", {k: len(v) for k, v in splits.items()})

## 5. Train

**Epoch 0** scores the starting model before any training. It also re-checks the model's
recorded ShanghaiTech results, and **stops with an explanation** if they don't match,
because training from a mismatched setup would make the model worse. Expect lines like:

```
ShanghaiTech A test    recorded MAE    72.84  measured    73.53  [ok]
ShanghaiTech B test    recorded MAE    13.33  measured    13.38  [ok]
```

The first epoch is the slowest, because it builds every ground-truth density map
(they're cached after that). A checkpoint is saved only when it beats the best
validation score so far, so whatever ends up on Drive is at least as good as the start.
If you see `CUDA out of memory`, stop and tell Claude.

In [ ]:
%cd /content/CENTINAL
!python -u train_csrnet.py \
    --data_root /content/data \
    --datasets {DATASETS} \
    --preprocess raw \
    --init_from {INIT_FROM} \
    --lr {LR} --epochs {EPOCHS} --eval_every {EVAL_EVERY} \
    --time_budget_min {TIME_BUDGET_MIN} \
    --workers 2 --cache_dir /content/density_cache \
    --out {CHECKPOINT}

### Resume (only if the session disconnected)

Rerun cells 1–4 first, then this cell. It continues from the best checkpoint on Drive.

In [ ]:
%cd /content/CENTINAL
!python -u train_csrnet.py \
    --data_root /content/data \
    --datasets {DATASETS} \
    --preprocess raw \
    --init_from {CHECKPOINT} \
    --lr {LR} --epochs {EPOCHS} --eval_every {EVAL_EVERY} \
    --time_budget_min {TIME_BUDGET_MIN} \
    --workers 2 --cache_dir /content/density_cache \
    --out {CHECKPOINT}

## 6. Score on the test sets

Runs the same benchmark script used for the README on every test set, for the new model
and (if `EVALUATE_OLD_MODEL`) the old one. JHU-CROWD++ has 1,600 test images, so this step
takes a while.

In [ ]:
%cd /content/CENTINAL
import os, subprocess, sys, torch
sys.path.insert(0, "/content/CENTINAL")
from centinal.datasets import find_dataset_root

if not os.path.exists(CHECKPOINT):
    raise RuntimeError(f"No checkpoint at {CHECKPOINT}: no epoch beat the starting model yet.")
meta = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
print("Best checkpoint is from epoch", meta["epoch"], "| validation score", round(meta["score"], 4))

TEST_SUBDIR = {"sha": "test_data", "shb": "test_data", "qnrf": "Test", "jhu": "test"}
names = [d.strip() for d in DATASETS.split(",")]
test_dirs = {n: os.path.join(find_dataset_root("/content/data", n), TEST_SUBDIR[n]) for n in names}
models = {"v2": CHECKPOINT}
if EVALUATE_OLD_MODEL:
    models["v1"] = "csrnet_centinal.pth"

for tag, model_path in models.items():
    for name in names:
        if tag == "v1" and name in ("sha", "shb"):
            continue   # already measured in the README
        print(f"\n=== {tag} on {name} ===", flush=True)
        subprocess.run([sys.executable, "eval/eval_csrnet.py", "--model_path", model_path,
                        "--dataset_path", test_dirs[name], "--tag", f"{name}_{tag}",
                        "--output_dir", f"{DRIVE_DIR}/results_v2"], check=False)

## 7. Recalibrate the risk classifier for the new model

The risk classifier's feature scaling and noise estimate depend on the counting model, so
it's retrained here against the new checkpoint. This takes a few minutes. Keep the result
only if the new counting model is adopted.

In [ ]:
%cd /content/CENTINAL
!python -u train_lstm.py --csrnet_path {CHECKPOINT} --epochs 80 --tracks 40 --out_dir {DRIVE_DIR}/lstm_v2

## 8. Bring it back

From `MyDrive/centinal_training/` in Google Drive, download:

- `csrnet_centinal_v2.pth` and `csrnet_centinal_v2_history.json`
- the `results_v2/` folder
- the `lstm_v2/` folder

Leave them in your Downloads folder and tell Claude. Claude will compare the models
and, if the new one is better, make it the default.